In [2]:
#%pip install scikit-learn
import pandas as pd
import numpy as np

In [3]:
sepsis_readm_severity=pd.read_csv("/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/sepsis_cohort_severity.csv")

In [ ]:

# from sklearn.linear_model import LogisticRegression

# 1. Clean your model data to drop missing values
model_data = sepsis_readm_severity.dropna(subset=[
    'readmitted_30d', 'drg_severity', 'has_mech_vent', 
    'has_high_cost_procedure'
])

# 2. Create dummy variables for categorical features (like drg_severity)
X = pd.get_dummies(
    model_data[['drg_severity', 'has_mech_vent', 'has_high_cost_procedure']], 
    drop_first=True
)
y = model_data['readmitted_30d']

# 3. Fit Logistic Regression with L2 Penalty (Ridge regularization) using scikit-learn
# penalty='l2' stops coefficients from blowing up to infinity due to separation
# C is the inverse of regularization strength (smaller C = stronger penalty)
lr_model = LogisticRegression(penalty='l2', C=1.0, solver='lbfgs')
lr_model.fit(X, y)

# 4. Calculate Standard Errors for Confidence Intervals (using the Hessian approximation)
# This calculates the inverse of the second derivative (Fisher Information Matrix)
X_design = np.hstack([np.ones((X.shape[0], 1)), X.values])
probabilities = lr_model.predict_proba(X_design[:, 1:])[:, 1]
# Weight matrix for logistic regression
W = probabilities * (1 - probabilities)
# Hessian matrix calculation
D = np.dot(X_design.T, X_design * W[:, np.newaxis])
# Add a small ridge penalty diagonal term matching C to prevent singular matrix errors
D += (1.0 / 1.0) * np.eye(D.shape[0]) 
cov_matrix = np.linalg.inv(D)
standard_errors = np.sqrt(np.diagonal(cov_matrix))[1:] # Exclude intercept

# 5. Compute 95% Confidence Intervals for Coefficients and Odds Ratios
coefs = lr_model.coef_[0]
z_score = 1.96  # 95% confidence level

ci_lower_coef = coefs - (z_score * standard_errors)
ci_upper_coef = coefs + (z_score * standard_errors)

# Exponentiate to get Odds Ratios and their respective Confidence Intervals
odds_ratios = np.exp(coefs)
ci_lower_or = np.exp(ci_lower_coef)
ci_upper_or = np.exp(ci_upper_coef)

# 6. Assemble into a clean results DataFrame
results_df = pd.DataFrame({
    'Variable': X.columns,
    'Odds_Ratio': odds_ratios,
    'CI_Lower': ci_lower_or,
    'CI_Upper': ci_upper_or,
    'Coefficient': coefs,
    'Std_Error': standard_errors
})

print(results_df)

results_df.to_csv('/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/sepsis_logistic_regression_results.csv', index=True)

In [4]:
# Get DRG codes for sepsis cohort to inquire their relative weights

unique_sepsis_drg=sepsis_readm_severity['drg_code'].unique().tolist()
print(unique_sepsis_drg)

[155, 314, 287, 628, 987, 25, 20, 207, 189, 193, 266, 326, 377, 371, 441, 420, 480, 853, 854, 872, 871, 870, 981]


In [6]:


# 1. Define CMS MS-DRG Relative Weights (FY 2027 Baseline)
# Update or expand this dictionary if your cohort includes additional DRGs
ms_drg_weights = {
    '020': 7.9629,
    '025': 4.6118,
    '155': 0.9322,
    '189': 1.2108,
    '193': 1.2825,
    '207': 6.3977,
    '266': 5.9309,
    '287': 1.0784,
    '314': 2.0873,
    '326': 4.9436,
    '371': 1.7089,
    '377': 1.8127,
    '420': 3.0664,
    '441': 1.8219,
    '480': 2.9204,
    '628': 3.7050,
    '853': 4.9306,
    '854': 2.0095,
    '870': 6.6634,
    '871': 1.9642,
    '872': 1.0785,
    '981': 4.7412,
    '987': 3.3515
}

# Standard CMS National Unadjusted Operating Base Rate (~$6,848 for FY 2027)
MEDICARE_BASE_RATE = 6848.00

# 2. Map weights and calculate estimated reimbursement per admission
sepsis_readm_severity['drg_code_str'] = sepsis_readm_severity['drg_code'].astype(str).str.strip()
sepsis_readm_severity['drg_weight'] = sepsis_readm_severity['drg_code_str'].map(ms_drg_weights)
sepsis_readm_severity['est_reimbursement'] = sepsis_readm_severity['drg_weight'] * MEDICARE_BASE_RATE

# 3. Summarize Readmission Volume & Financial Exposure by MS-DRG
financial_summary = (
    sepsis_readm_severity.dropna(subset=['drg_weight'])
    .groupby('drg_code_str')
    .agg(
        total_admissions=('readmitted_30d', 'count'),
        readmissions_30d=('readmitted_30d', 'sum'),
        readmission_rate=('readmitted_30d', lambda x: round(x.mean() * 100, 2)),
        avg_case_payment=('est_reimbursement', 'mean'),
        total_readmission_spend=('est_reimbursement', lambda x: x[sepsis_readm_severity.loc[x.index, 'readmitted_30d'] == 1].sum())
    )
    .reset_index()
)

# Format currency columns for clean reporting
financial_summary['avg_case_payment_fmt'] = financial_summary['avg_case_payment'].map('${:,.2f}'.format)
financial_summary['total_readmission_spend_fmt'] = financial_summary['total_readmission_spend'].map('${:,.2f}'.format)

print("=== MS-DRG 30-Day Readmission Financial Exposure ===")
print(financial_summary[['drg_code_str', 'total_admissions', 'readmissions_30d', 
                         'readmission_rate', 'avg_case_payment_fmt', 'total_readmission_spend_fmt']])

# Export summary CSV to feed directly into your Tableau dashboard
financial_summary.to_csv('/Users/xumuyao/Desktop/JOBHUNTING/mockdata/hosp/ms_drg_financial_impact.csv', index=False)


=== MS-DRG 30-Day Readmission Financial Exposure ===
   drg_code_str  total_admissions  readmissions_30d  readmission_rate  \
0           155                 1                 0              0.00   
1           189                 1                 1            100.00   
2           193                 1                 1            100.00   
3           207                 3                 1             33.33   
4           266                 1                 1            100.00   
5           287                 1                 0              0.00   
6           314                 1                 0              0.00   
7           326                 1                 0              0.00   
8           371                 1                 0              0.00   
9           377                 1                 0              0.00   
10          420                 1                 0              0.00   
11          441                 1                 0              0.00  